In [1]:
import sys 

sys.path.append("..")

In [2]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

In [3]:
from model.aecr_net import AECRNet
from model.aod_net import AODNet
from model.griddehaze_net import GridDehazeNet
from model.ffa_net import FFA
from model.msbdn import MSBDN
from model.dehamer import Dehamer 
from model.fsdgn import FSDGN
from model.fmphys_mamba import FM_PhysMamba_UNET, ODESolver 
from model.dehazeddpm import MPRfusion, UNet, GaussianDiffusion
from model.dehazeddpm import make_beta_schedule

In [4]:
from thop import clever_format
from ptflops import get_model_complexity_info

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
def benchmark_memory(model, input_data, mode='inference'):
    """
    Measures peak memory for either a forward pass (inference) 
    or forward + backward pass (training).
    """
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    if mode == 'inference':
        with torch.no_grad():
            if isinstance(input_data, dict):
                output = model(**input_data)
            else:
                output = model(input_data)
    
    elif mode == 'training':
        model.train()
        # Ensure inputs require grad if they are tensors
        if isinstance(input_data, dict):
            output = model(**input_data)
        else:
            output = model(input_data)
        
        # Simulate a loss and backward pass
        # If output is a dict (like some Diffusion models), get the tensor
        target = torch.randn_like(output if torch.is_tensor(output) else output[0])
        loss = torch.nn.functional.mse_loss(output if torch.is_tensor(output) else output[0], target)
        loss.backward()
    
    peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return f"{peak_mem:.2f} MB"

In [7]:
# 1. Initialize all models
models = {
    "AOD-Net": AODNet().to(device),
    "GridDehazeNet": GridDehazeNet().to(device),
    "AECR-Net": AECRNet().to(device),
    "FFA-Net": FFA().to(device),
    "MSBDN": MSBDN().to(device),
    "FSDGN": FSDGN().to(device),
}

results = []

def format_to_gflops(raw_ops):
    return f"{raw_ops / 1e9:,.2f} GFLOPs"

# 2. Calculate for standard single-stage models
for name, model in models.items():
    print(name)
    input_hazy = torch.randn(1, 3, 256, 256).to(device)
    macs, params = get_model_complexity_info(
        model, (3, 256, 256), 
        as_strings=False, 
        print_per_layer_stat=False, 
        verbose=False
    )
    _, p_fmt = clever_format([macs, params], "%.2f")
    
    inf_mem = benchmark_memory(model, input_hazy, mode='inference')
    train_mem = benchmark_memory(model, input_hazy, mode='training')
    
    results.append({
        "Model": name, 
        "Params": p_fmt, 
        "GFLOPs": format_to_gflops(macs),
        "Inference Memory": inf_mem,
        "Train Memory": train_mem
    })

AOD-Net
GridDehazeNet
AECR-Net
FFA-Net
MSBDN
FSDGN


In [8]:
# Flow Matching Calculation
fm_model = FM_PhysMamba_UNET("../configs/model_cfgs/small.yaml").to(device)
fm_nfe = 10 

# Define input constructor because FM models usually take (x, t)
def fm_input_constructor(input_res):
    return {
        'x': torch.randn(1, 3, 256, 256).to(device), 
        't': torch.randn(1).to(device)
    }

fm_macs_raw, fm_params_raw = get_model_complexity_info(
    fm_model, (3, 256, 256),
    input_constructor=fm_input_constructor,
    as_strings=False,
    print_per_layer_stat=False, 
    verbose=False
)

# Total Inference Cost = Base Cost * NFE
total_fm_ops = fm_macs_raw * fm_nfe
_, fm_p_fmt = clever_format([total_fm_ops, fm_params_raw], "%.2f")

fm_inputs = fm_input_constructor((3, 256, 256))

inf_mem = benchmark_memory(fm_model, fm_inputs, mode='inference')
train_mem = benchmark_memory(fm_model, fm_inputs, mode='training')

results.append({
    "Model": f"FM-PhysMamba (NFE={fm_nfe})", 
    "Params": fm_p_fmt, 
    "GFLOPs": format_to_gflops(total_fm_ops),
    "Inference Memory": inf_mem,
    "Train Memory": train_mem
})

Warning! No positional inputs found for a module, assuming batch size is 1.


In [9]:
# 4. Calculate for 2-Stage DehazeDDPM (T=1000)
# (Previous logic for DDPM remains valid for comparison)
ddpm_steps = 1000
s1_model = MPRfusion().to(device)
s1_macs, s1_params = get_model_complexity_info(
    s1_model, 
    (3, 256, 256), 
    as_strings=False, 
    print_per_layer_stat=False, 
    verbose=False
)
unet_model = UNet(in_channel=7, out_channel=3, inner_channel=64, image_size=256).to(device)
s2_macs, s2_params = get_model_complexity_info(
    unet_model, (7, 256, 256), 
    print_per_layer_stat=False, 
    input_constructor=lambda x: {'x': torch.randn(1, 7, 256, 256).to(device), 'time': torch.randn(1).to(device)}, 
    as_strings=False, verbose=False
)
total_ddpm_ops = s1_macs + (s2_macs * ddpm_steps)
total_ddpm_params = s1_params + s2_params
_, ddpm_p_fmt = clever_format([total_ddpm_ops, total_ddpm_params], "%.2f")

# SInce Stage 2 is the largest part, we report memory of the denoising step here
ddpm_inputs = {'x': torch.randn(1, 7, 256, 256).to(device), 'time': torch.randn(1).to(device)}

inf_mem = benchmark_memory(unet_model, ddpm_inputs, mode='inference')
train_mem = benchmark_memory(unet_model, ddpm_inputs, mode='training')

results.append({
    "Model": f"DehazeDDPM (T={ddpm_steps})", 
    "Params": ddpm_p_fmt, 
    "GFLOPs": format_to_gflops(total_ddpm_ops),
    "Inference Memory": inf_mem,
    "Train Memory": train_mem
})

# 5. Display the Final Organized Table
print("\n" + "="*80)
print(f"{'Model Name':<30} | {'Parameters':<15} | {'Complexity (GFLOPs)':<20} | {'Inference Memory':<20} | {'Train Memory':<20}")
print("-" * 80)

for res in results:
    print(f"{res['Model']:<30} | {res['Params']:<15} | {res['GFLOPs']:<20} | {res['Inference Memory']:<20} | {res['Train Memory']:<20}")

print("="*80)

Warning! No positional inputs found for a module, assuming batch size is 1.

Model Name                     | Parameters      | Complexity (GFLOPs)  | Inference Memory     | Train Memory        
--------------------------------------------------------------------------------
AOD-Net                        | 1.76K           | 0.12 GFLOPs          | 169.90 MB            | 247.99 MB           
GridDehazeNet                  | 1.78M           | 6.30 GFLOPs          | 188.74 MB            | 334.63 MB           
AECR-Net                       | 766.35K         | 4.32 GFLOPs          | 263.59 MB            | 319.30 MB           
FFA-Net                        | 4.46M           | 288.86 GFLOPs        | 296.80 MB            | 4132.25 MB          
MSBDN                          | 31.35M          | 24.58 GFLOPs         | 245.50 MB            | 725.55 MB           
FSDGN                          | 2.73M           | 10.74 GFLOPs         | 386.49 MB            | 863.73 MB           
FM-PhysMamba (NF